# 02 - Train YOLOv8 fracture detector

Run this notebook on a GPU runtime in Colab or Kaggle. Training happens here, not in server scripts.

Inputs:
- A small fracture-only YOLO dataset from `01_dataset_setup.ipynb`.
- A dataset YAML whose only class is `0: fracture`.

Outputs:
- Training runs under `runs/fracture/train`.
- Best weights copied to `weights/fracture_yolov8n_best.pt`.
- In Colab, both folders are inside Google Drive so they persist after the runtime stops.


In [ ]:
RUN_ENV = "local"  # "colab" or "kaggle" recommended for GPU training
PROJECT_NAME = "yolov8-fracture-detection"

MODEL_NAME = "yolov8n.pt"
EPOCHS = 30
IMAGE_SIZE = 640
BATCH_SIZE = 16


In [ ]:
from pathlib import Path
import shutil
if RUN_ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_NAME
elif RUN_ENV == "kaggle":
    PROJECT_ROOT = Path("/kaggle/working") / PROJECT_NAME
else:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

DATA_YAML = PROJECT_ROOT / "data" / "fracture_subset" / "fracture.yaml"
RUNS_ROOT = PROJECT_ROOT / "runs"
WEIGHTS_ROOT = PROJECT_ROOT / "weights"
WEIGHTS_ROOT.mkdir(parents=True, exist_ok=True)


print(f"Dataset YAML: {DATA_YAML}")
print(f"Runs root: {RUNS_ROOT}")
print(f"Weights root: {WEIGHTS_ROOT}")


In [ ]:
# Install inside hosted notebook runtimes when needed.
# In local development, prefer installing dependencies from pyproject.toml instead.
if RUN_ENV in {"colab", "kaggle"}:
    %pip install -U ultralytics roboflow


In [ ]:
from ultralytics import YOLO
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("Switch to a GPU runtime before serious training.")


In [ ]:
from dataclasses import dataclass

IMAGE_EXTENSIONS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}

@dataclass(frozen=True)
class DatasetCounts:
    train_images: int
    val_images: int
    test_images: int
    train_labels: int
    val_labels: int
    test_labels: int


def summarize_yolo_dataset(dataset_root):
    root = Path(dataset_root).expanduser()
    counts = []
    for kind in ("images", "labels"):
        for split in ("train", "val", "test"):
            folder = root / kind / split
            if kind == "images":
                counts.append(sum(1 for item in folder.glob("*") if item.suffix.lower() in IMAGE_EXTENSIONS))
            else:
                counts.append(sum(1 for item in folder.glob("*.txt")))
    return DatasetCounts(*counts)


def validate_single_class_labels(dataset_root):
    root = Path(dataset_root).expanduser()
    bad_lines = []
    for label_path in sorted((root / "labels").glob("**/*.txt")):
        for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
            stripped = line.strip()
            if not stripped:
                continue
            parts = stripped.split()
            if len(parts) != 5 or parts[0] != "0":
                bad_lines.append(f"{label_path}:{line_number}: {stripped}")
    if bad_lines:
        preview = "\n".join(bad_lines[:10])
        raise ValueError(f"Labels must be YOLO detect rows with class id 0 only:\n{preview}")

if not DATA_YAML.exists():
    raise FileNotFoundError(f"Run 01_dataset_setup.ipynb first or update DATA_YAML: {DATA_YAML}")

dataset_root = DATA_YAML.parent
validate_single_class_labels(dataset_root)
print(summarize_yolo_dataset(dataset_root))
print(DATA_YAML.read_text())


In [ ]:
model = YOLO(MODEL_NAME)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    project=str(RUNS_ROOT / "fracture"),
    name="train",
    exist_ok=True,
    seed=42,
)


In [ ]:
best_weight = RUNS_ROOT / "fracture" / "train" / "weights" / "best.pt"
final_weight = WEIGHTS_ROOT / "fracture_yolov8n_best.pt"
if not best_weight.exists():
    raise FileNotFoundError(f"Training did not produce best.pt at {best_weight}")
shutil.copy2(best_weight, final_weight)
print(f"Saved persistent weights to: {final_weight}")


## Next step

Open `03_evaluate_predict.ipynb` and set `WEIGHTS_PATH` to the saved path printed above.
